# Stage 5 - Frozen risk-development inference and baseline analysis

This notebook evaluates the two frozen publication agents on the locked
`risk_dev` cohort. Disease probabilities are the unweighted arithmetic mean
of the two agents' softmax probabilities. It does not train a consensus model,
tune a threshold, fit a calibrator, access the locked test cohort, or compute
saliency/attention features.

Runtime requires explicit image files through `ARGUS_IMAGE_DIR`. Set
`ARGUS_SMOKE_TEST=1` for the diagnostic subset (at most 16 images); smoke
metrics are implementation diagnostics and are not scientific results.


In [ ]:
# Standard-library preflight. Scientific packages are intentionally imported later,
# after checkpoint metadata and recorded package versions have been inspected.
import ast
import contextlib
import dataclasses
import datetime as dt
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

FORBIDDEN_SPLIT_TOKEN = "final" + "_test"
REQUIRED_SPLIT_FILES = (
    "train.csv",
    "model_val.csv",
    "risk_dev.csv",
    "dataset_fingerprint.json",
    "split_hashes.json",
)
CHECKPOINT_ARCHIVES = {
    "agent_a": ("agent_a_checkpoint.zip", "agent_a_best.pth", "agent_a_run_metadata.json"),
    "agent_b": ("agent_b_checkpoint.zip", "agent_b_best.pth", "agent_b_run_metadata.json"),
}


def _find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    notebook_hint = Path("ml_training")
    for candidate in candidates:
        if (candidate / notebook_hint / "config.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the Argus Vision repository root.")


def _require_kaggle_environment(name):
    value = os.environ.get(name, "").strip()
    if not value:
        raise RuntimeError(f"{name} must be set explicitly on Kaggle.")
    return Path(value).expanduser().resolve()


def _assert_not_forbidden_path(path, purpose):
    text = str(path).lower()
    if FORBIDDEN_SPLIT_TOKEN in text:
        raise RuntimeError(f"Forbidden locked-test path requested for {purpose}.")


def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _validate_zip_member_name(name):
    normalized = name.replace("\\", "/")
    pure = PurePosixPath(normalized)
    if not normalized or pure.is_absolute() or ".." in pure.parts or ":" in pure.parts[0]:
        raise RuntimeError("Unsafe ZIP member path detected.")
    return normalized


def _required_zip_members(zip_path, required_names):
    matches = {name: [] for name in required_names}
    seen = set()
    with zipfile.ZipFile(zip_path, "r") as archive:
        for info in archive.infolist():
            normalized = _validate_zip_member_name(info.filename)
            folded = normalized.casefold()
            if folded in seen:
                raise RuntimeError(f"Duplicate ZIP member name in {zip_path.name}: {normalized}")
            seen.add(folded)
            basename = PurePosixPath(normalized).name
            for required in required_names:
                if basename.casefold() == required.casefold():
                    matches[required].append(info.filename)
    bad = {name: values for name, values in matches.items() if len(values) != 1}
    if bad:
        raise RuntimeError(f"ZIP {zip_path.name} does not contain exactly one of each required member: {bad}")
    return {name: values[0] for name, values in matches.items()}


def _read_metadata_from_checkpoint_zip(zip_path, metadata_name):
    members = _required_zip_members(zip_path, (metadata_name,))
    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(members[metadata_name], "r") as handle:
            return json.loads(handle.read().decode("utf-8"))


def _resolve_split_source(repo_root, is_kaggle):
    override = os.environ.get("ARGUS_SPLIT_DIR", "").strip()
    if is_kaggle:
        candidates = [_require_kaggle_environment("ARGUS_SPLIT_DIR")]
    elif override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        candidates = [
            repo_root / "artifacts" / "rescue" / "splits",
            repo_root / "artifacts" / "splits",
            repo_root / "argus-publication-splits",
            repo_root / "argus-publication-splits.zip",
        ]
    checked = []
    for candidate in candidates:
        _assert_not_forbidden_path(candidate, "split source")
        if candidate.is_dir():
            complete = all((candidate / name).is_file() for name in REQUIRED_SPLIT_FILES)
            checked.append({"path": str(candidate), "kind": "directory", "complete": complete})
            if complete:
                return candidate, "directory", checked
        elif candidate.is_file() and candidate.suffix.lower() == ".zip":
            try:
                _required_zip_members(candidate, REQUIRED_SPLIT_FILES)
                checked.append({"path": str(candidate), "kind": "zip", "complete": True})
                return candidate, "zip", checked
            except Exception as exc:
                checked.append({"path": str(candidate), "kind": "zip", "complete": False,
                                "reason": type(exc).__name__})
        else:
            checked.append({"path": str(candidate), "kind": "missing", "complete": False})
    raise FileNotFoundError(f"No unambiguous publication split source found. Checked: {checked}")


def _package_preflight(metadata_by_agent):
    versions_a = metadata_by_agent["agent_a"].get("package_versions", {})
    versions_b = metadata_by_agent["agent_b"].get("package_versions", {})
    if versions_a != versions_b:
        raise RuntimeError("Agent checkpoint metadata records incompatible package environments.")
    distribution_names = {
        "python": None,
        "torch": "torch",
        "torchvision": "torchvision",
        "timm": "timm",
        "numpy": "numpy",
        "pandas": "pandas",
        "scikit-learn": "scikit-learn",
        "Pillow": "Pillow",
        "torchmetrics": "torchmetrics",
        "matplotlib": "matplotlib",
        "pyarrow": "pyarrow",
    }
    report = {}
    missing_required = []
    required_runtime = {"torch", "torchvision", "timm", "numpy", "pandas",
                        "scikit-learn", "Pillow", "matplotlib", "pyarrow"}
    for logical, distribution in distribution_names.items():
        recorded = versions_a.get(logical)
        if logical == "python":
            runtime = platform.python_version()
        else:
            try:
                runtime = importlib.metadata.version(distribution)
            except importlib.metadata.PackageNotFoundError:
                runtime = None
        status = "missing" if runtime is None else ("match" if recorded == runtime else "mismatch")
        if recorded is None and runtime is not None:
            status = "runtime_only_unrecorded"
        report[logical] = {"recorded": recorded, "runtime": runtime, "status": status}
        if logical in required_runtime and runtime is None:
            missing_required.append(logical)
    timm_recorded = versions_a.get("timm")
    timm_runtime = report["timm"]["runtime"]
    if timm_recorded != "1.0.28":
        raise RuntimeError(f"Expected recorded timm 1.0.28, found {timm_recorded!r} in metadata.")
    if timm_runtime != "1.0.28":
        raise RuntimeError(
            f"Publication checkpoint reconstruction requires timm==1.0.28; installed={timm_runtime!r}. "
            "The notebook will not install, replace, or upgrade it."
        )
    if missing_required:
        raise RuntimeError(
            "Missing required runtime packages: " + ", ".join(sorted(missing_required)) +
            ". Install only the missing packages before executing the notebook."
        )
    return report, versions_a


REPO_ROOT = _find_repo_root()
IS_KAGGLE = Path("/kaggle").exists()
SMOKE_TEST = os.environ.get("ARGUS_SMOKE_TEST", "0") == "1"
if IS_KAGGLE:
    CHECKPOINT_DIR = _require_kaggle_environment("ARGUS_CHECKPOINT_DIR")
    ARTIFACT_ROOT = Path(os.environ.get("ARGUS_ARTIFACT_ROOT", "/kaggle/working/argus-stage5-artifacts")).resolve()
else:
    CHECKPOINT_DIR = Path(os.environ.get("ARGUS_CHECKPOINT_DIR", REPO_ROOT / "artifacts")).resolve()
    ARTIFACT_ROOT = Path(os.environ.get("ARGUS_ARTIFACT_ROOT", REPO_ROOT / "artifacts")).resolve()
SPLIT_SOURCE, SPLIT_SOURCE_KIND, SPLIT_PATH_REPORT = _resolve_split_source(REPO_ROOT, IS_KAGGLE)
IMAGE_DIR_ENV = os.environ.get("ARGUS_IMAGE_DIR", "").strip()
OUTPUT_DIR = ARTIFACT_ROOT / "rescue" / "risk_dev" / ("smoke" if SMOKE_TEST else "")
OUTPUT_DIR = OUTPUT_DIR.resolve()
_assert_not_forbidden_path(OUTPUT_DIR, "artifact output")

CHECKPOINT_PATHS = {}
CHECKPOINT_METADATA = {}
ARCHIVE_HASHES = {}
for agent, (archive_name, _checkpoint_name, metadata_name) in CHECKPOINT_ARCHIVES.items():
    archive_path = (CHECKPOINT_DIR / archive_name).resolve()
    _assert_not_forbidden_path(archive_path, "checkpoint archive")
    if not archive_path.is_file():
        raise FileNotFoundError(f"Missing immutable checkpoint archive: {archive_path}")
    CHECKPOINT_PATHS[agent] = archive_path
    ARCHIVE_HASHES[agent] = _sha256_file(archive_path)
    CHECKPOINT_METADATA[agent] = _read_metadata_from_checkpoint_zip(archive_path, metadata_name)

PACKAGE_COMPATIBILITY, RECORDED_PACKAGE_VERSIONS = _package_preflight(CHECKPOINT_METADATA)

print("Stage 5 path-resolution report")
print("  repository root    :", REPO_ROOT)
print("  Agent A archive    :", CHECKPOINT_PATHS["agent_a"])
print("  Agent B archive    :", CHECKPOINT_PATHS["agent_b"])
print("  split source       :", SPLIT_SOURCE, f"({SPLIT_SOURCE_KIND})")
print("  artifact root      :", ARTIFACT_ROOT)
print("  smoke test         :", SMOKE_TEST)
print("  image directory set:", bool(IMAGE_DIR_ENV))


In [ ]:
# Scientific imports occur only after the metadata-driven package preflight above.
# Load torch before pyarrow on Windows to avoid conflicting DLL initialization order.
import torch
from torch.utils.data import DataLoader, Dataset
import timm
from torchvision import transforms

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import pyarrow  # noqa: F401 - required for the Parquet contract
import sklearn
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)

sys.path.insert(0, str(REPO_ROOT / "ml_training"))
from config import IMAGE_SIZE, IMAGENET_MEAN, IMAGENET_STD, ISIC_CLASSES, NUM_CLASSES
from transforms import get_eval_transform

EXPECTED_CLASSES = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]
EXPECTED_ARCHITECTURES = {
    "agent_a": "efficientnet_b4",
    "agent_b": "vit_base_patch16_224.augreg_in21k_ft_in1k",
}
BOOTSTRAP_SEED = 2026
BOOTSTRAP_REPLICATES = 50 if SMOKE_TEST else 2000
COVERAGE_TARGETS = (1.00, 0.95, 0.90, 0.80, 0.70)
RISK_SCORE_COLUMNS = (
    "risk_msp",
    "risk_normalized_entropy",
    "risk_margin",
    "risk_normalized_js",
    "risk_dual_uncertainty",
)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


In [ ]:
@contextlib.contextmanager
def materialize_split_source(source, source_kind):
    if source_kind == "directory":
        paths = {name: source / name for name in REQUIRED_SPLIT_FILES}
        yield paths
        return
    members = _required_zip_members(source, REQUIRED_SPLIT_FILES)
    with tempfile.TemporaryDirectory(prefix="argus_stage5_splits_") as tmp:
        root = Path(tmp)
        paths = {}
        with zipfile.ZipFile(source, "r") as archive:
            for name in REQUIRED_SPLIT_FILES:
                destination = root / name
                with archive.open(members[name], "r") as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
                paths[name] = destination
        yield paths


def _safe_split_hash_manifest(path):
    def keep_allowed_pairs(pairs):
        return {
            key: value for key, value in pairs
            if FORBIDDEN_SPLIT_TOKEN not in str(key).lower()
        }
    with Path(path).open("r", encoding="utf-8") as handle:
        result = json.load(handle, object_pairs_hook=keep_allowed_pairs)
    if any(FORBIDDEN_SPLIT_TOKEN in str(key).lower() for key in result):
        raise RuntimeError("Forbidden split key survived manifest filtering.")
    return result


def normalize_image_id(value):
    text = str(value).strip()
    text = os.path.basename(text)
    for extension in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"):
        if text.lower().endswith(extension):
            text = text[:-len(extension)]
            break
    return text.lower()


def _present_strings(series):
    return {str(value).strip() for value in series if pd.notna(value) and str(value).strip()}


def _load_and_validate_splits(paths, metadata_by_agent):
    hashes = _safe_split_hash_manifest(paths["split_hashes.json"])
    actual_hashes = {}
    for filename in REQUIRED_SPLIT_FILES[:-2]:
        expected = hashes.get(filename)
        actual = _sha256_file(paths[filename])
        if not expected or str(expected).lower() != actual:
            raise RuntimeError(f"Split hash mismatch for {filename}.")
        actual_hashes[filename] = actual
    fingerprint_hash = _sha256_file(paths["dataset_fingerprint.json"])
    with paths["dataset_fingerprint.json"].open("r", encoding="utf-8") as handle:
        fingerprint = json.load(handle)
    if fingerprint.get("split_protocol") != "publication_rescue_v1":
        raise RuntimeError("Unexpected publication split protocol.")
    if fingerprint.get("cohort_policy") != "verified_lesion_only":
        raise RuntimeError("Stage 5 requires the verified-lesion-only publication cohort.")

    frames = {
        "train": pd.read_csv(paths["train.csv"]),
        "model_val": pd.read_csv(paths["model_val.csv"]),
        "risk_dev": pd.read_csv(paths["risk_dev.csv"]),
    }
    required_columns = {"image", "label", "lesion_id", "lesion_group", "research_split"}
    for split_name, frame in frames.items():
        missing = sorted(required_columns - set(frame.columns))
        if missing:
            raise RuntimeError(f"{split_name} is missing required columns: {missing}")
        values = set(frame["research_split"].dropna().astype(str))
        if values != {split_name}:
            raise RuntimeError(f"Invalid research_split values in {split_name}: {sorted(values)}")
        normalized = frame["image"].map(normalize_image_id)
        if normalized.duplicated().any():
            raise RuntimeError(f"Duplicate normalized image identifiers in {split_name}.")

    risk = frames["risk_dev"]
    if risk.empty:
        raise RuntimeError("risk_dev is empty.")
    bad_lesion = risk["lesion_id"].isna() | risk["lesion_id"].astype(str).str.strip().eq("")
    bad_group = risk["lesion_group"].isna() | risk["lesion_group"].astype(str).str.strip().eq("")
    expected_groups = "L:" + risk["lesion_id"].astype(str).str.strip()
    inconsistent = risk["lesion_group"].astype(str).str.strip().ne(expected_groups)
    if bad_lesion.any() or bad_group.any() or inconsistent.any():
        raise RuntimeError(
            "risk_dev violates the verified lesion_group contract: "
            f"missing_lesion_id={int(bad_lesion.sum())}, missing_group={int(bad_group.sum())}, "
            f"inconsistent_group={int(inconsistent.sum())}."
        )

    overlap_report = {}
    for other_name in ("train", "model_val"):
        other = frames[other_name]
        overlaps = {
            "image": len(set(risk["image"].map(normalize_image_id)) & set(other["image"].map(normalize_image_id))),
            "lesion_id": len(_present_strings(risk["lesion_id"]) & _present_strings(other["lesion_id"])),
            "lesion_group": len(_present_strings(risk["lesion_group"]) & _present_strings(other["lesion_group"])),
        }
        overlap_report[f"risk_dev__{other_name}"] = overlaps
        if any(overlaps.values()):
            raise RuntimeError(f"Publication split overlap detected against {other_name}: {overlaps}")

    labels = pd.to_numeric(risk["label"], errors="raise").astype(int)
    if not labels.between(0, NUM_CLASSES - 1).all():
        raise RuntimeError("risk_dev labels are outside the canonical eight-class encoding.")
    risk = risk.copy()
    risk["label"] = labels
    risk["_normalized_image_id"] = risk["image"].map(normalize_image_id)

    embedded_a = metadata_by_agent["agent_a"].get("dataset_fingerprint")
    embedded_b = metadata_by_agent["agent_b"].get("dataset_fingerprint")
    if embedded_a != embedded_b or embedded_a != fingerprint:
        raise RuntimeError("Checkpoint and split dataset fingerprints are incompatible.")
    for agent, metadata in metadata_by_agent.items():
        if str(metadata.get("dataset_fingerprint_file_sha256", "")).lower() != fingerprint_hash:
            raise RuntimeError(f"{agent} fingerprint-file hash is incompatible.")
        recorded_splits = metadata.get("split_manifest_hashes", {})
        for filename in ("train.csv", "model_val.csv"):
            if str(recorded_splits.get(filename, "")).lower() != actual_hashes[filename]:
                raise RuntimeError(f"{agent} metadata is incompatible with {filename}.")
    return risk.reset_index(drop=True), frames, fingerprint, fingerprint_hash, actual_hashes, overlap_report


def _assignment_from_python_source(source, name):
    tree = ast.parse(source)
    for node in tree.body:
        if isinstance(node, (ast.Assign, ast.AnnAssign)):
            targets = node.targets if isinstance(node, ast.Assign) else [node.target]
            if any(isinstance(target, ast.Name) and target.id == name for target in targets):
                return ast.literal_eval(node.value)
    raise RuntimeError(f"Could not find assignment {name!r}.")


def _assignment_from_notebook_text(text, name):
    notebook = json.loads(text)
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", [])) if isinstance(cell.get("source"), list) else cell.get("source", "")
        try:
            return _assignment_from_python_source(source, name)
        except (SyntaxError, ValueError, RuntimeError):
            continue
    raise RuntimeError(f"Could not find notebook assignment {name!r}.")


def verify_class_order_provenance(metadata_by_agent, labels):
    config_source = (REPO_ROOT / "ml_training" / "config.py").read_text(encoding="utf-8")
    config_order = _assignment_from_python_source(config_source, "ISIC_CLASSES")
    if list(config_order) != EXPECTED_CLASSES or list(ISIC_CLASSES) != EXPECTED_CLASSES:
        raise RuntimeError("config.py class order is not the publication ISIC-8 contract.")
    provenance = {
        "class_order": EXPECTED_CLASSES,
        "class_order_embedded_in_checkpoint_metadata": False,
        "config_py": {"verified": True, "path": str(REPO_ROOT / "ml_training" / "config.py")},
        "split_label_encoding": {
            "verified": set(map(int, np.unique(labels))).issubset(set(range(NUM_CLASSES))),
            "observed_indices": sorted(map(int, np.unique(labels))),
        },
        "historical_training_notebooks": {},
    }
    notebook_paths = {"agent_a": "ml_training/01_train_agent_a.ipynb",
                      "agent_b": "ml_training/02_train_agent_b.ipynb"}
    for agent, notebook_path in notebook_paths.items():
        commit = str(metadata_by_agent[agent].get("git_commit", "")).strip()
        record = {"git_commit": commit, "path": notebook_path, "inspected": False, "verified": False}
        try:
            text = subprocess.check_output(
                ["git", "show", f"{commit}:{notebook_path}"],
                cwd=REPO_ROOT, text=True, stderr=subprocess.STDOUT,
            )
            order = _assignment_from_notebook_text(text, "ISIC_CLASSES")
            architecture = _assignment_from_notebook_text(text, "MODEL_NAME")
            record.update({"inspected": True, "verified": list(order) == EXPECTED_CLASSES,
                           "architecture": architecture})
            if not record["verified"] or architecture != EXPECTED_ARCHITECTURES[agent]:
                raise RuntimeError(f"Historical {agent} notebook contract mismatch.")
        except subprocess.CalledProcessError as exc:
            record["limitation"] = "Historical training commit could not be inspected."
        provenance["historical_training_notebooks"][agent] = record
    return provenance


In [ ]:
def resolve_image_paths(frame, image_dir):
    root = Path(image_dir).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"ARGUS_IMAGE_DIR is not a directory: {root}")
    requested = set(frame["_normalized_image_id"].astype(str))
    matches = defaultdict(list)

    def consider(path, allowed_ids):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            return
        stem = normalize_image_id(path.name)
        if stem in allowed_ids:
            matches[stem].append(path.resolve())

    for path in root.iterdir():
        consider(path, requested)
    unresolved = requested - set(matches)
    if unresolved:
        for path in root.rglob("*"):
            if path.parent == root:
                continue
            consider(path, unresolved)

    duplicate = {key: values for key, values in matches.items() if len(values) > 1}
    if duplicate:
        details = {key: [str(path) for path in values] for key, values in duplicate.items()}
        raise RuntimeError(f"Image identifiers map to multiple files under ARGUS_IMAGE_DIR={root}: {details}")
    missing = sorted(requested - set(matches))
    if missing:
        reports = [
            {"image_identifier": identifier,
             "constructed_path": str(root / f"{identifier}.jpg"),
             "ARGUS_IMAGE_DIR": str(root)}
            for identifier in missing
        ]
        raise FileNotFoundError(f"Selected images are missing; no rows were skipped: {reports}")
    result = frame.copy()
    result["image_path"] = result["_normalized_image_id"].map(lambda key: str(matches[key][0]))
    return result


def select_smoke_subset(frame):
    ordered = frame.sort_values(["label", "_normalized_image_id"], kind="mergesort")
    rows = []
    for label in range(NUM_CLASSES):
        seen_groups = set()
        for index, row in ordered.loc[ordered["label"] == label].iterrows():
            group = str(row["lesion_group"])
            if group in seen_groups:
                continue
            seen_groups.add(group)
            rows.append(index)
            if len(seen_groups) == 2:
                break
    subset = frame.loc[rows].sort_values(["label", "_normalized_image_id"], kind="mergesort")
    subset = subset.reset_index(drop=True)
    if len(subset) > 16 or subset["lesion_group"].duplicated().any():
        raise RuntimeError("Smoke subset violates the one-row-per-group, at-most-16 contract.")
    return subset


@contextlib.contextmanager
def extracted_checkpoint_inputs(checkpoint_paths):
    with tempfile.TemporaryDirectory(prefix="argus_stage5_checkpoints_") as tmp:
        root = Path(tmp)
        extracted = {}
        hashes = {}
        for agent, archive_path in checkpoint_paths.items():
            _archive_name, checkpoint_name, metadata_name = CHECKPOINT_ARCHIVES[agent]
            members = _required_zip_members(archive_path, (checkpoint_name, metadata_name))
            extracted[agent] = {}
            hashes[agent] = {}
            with zipfile.ZipFile(archive_path, "r") as archive:
                for logical, filename in (("checkpoint", checkpoint_name), ("metadata", metadata_name)):
                    destination = root / f"{agent}_{filename}"
                    with archive.open(members[filename], "r") as src, destination.open("wb") as dst:
                        shutil.copyfileobj(src, dst)
                    extracted[agent][logical] = destination
                    hashes[agent][logical] = _sha256_file(destination)
            expected = str(CHECKPOINT_METADATA[agent].get("checkpoint_sha256", "")).lower()
            if hashes[agent]["checkpoint"] != expected:
                raise RuntimeError(f"{agent} checkpoint member hash does not match run metadata.")
        yield extracted, hashes, root


def _classifier_out_features(model):
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else None
    if hasattr(classifier, "out_features"):
        return int(classifier.out_features)
    raise RuntimeError("Could not verify model classifier output dimension.")


def reconstruct_and_load_models(extracted, device="cpu"):
    models = {}
    report = {}
    for agent, architecture in EXPECTED_ARCHITECTURES.items():
        metadata_arch = CHECKPOINT_METADATA[agent].get("model_architecture")
        if metadata_arch != architecture:
            raise RuntimeError(f"{agent} metadata architecture mismatch: {metadata_arch!r}")
        model = timm.create_model(architecture, pretrained=False, num_classes=NUM_CLASSES)
        state = torch.load(extracted[agent]["checkpoint"], map_location="cpu", weights_only=True)
        if not isinstance(state, dict):
            raise RuntimeError(f"{agent} checkpoint is not a state dictionary.")
        incompatible = model.load_state_dict(state, strict=True)
        if incompatible.missing_keys or incompatible.unexpected_keys:
            raise RuntimeError(f"{agent} strict checkpoint loading returned incompatible keys.")
        output_dimension = _classifier_out_features(model)
        if output_dimension != NUM_CLASSES:
            raise RuntimeError(f"{agent} output dimension is {output_dimension}, expected {NUM_CLASSES}.")
        model.requires_grad_(False)
        model.eval()
        model.to(device)
        if any(parameter.requires_grad for parameter in model.parameters()):
            raise RuntimeError(f"{agent} has trainable parameters after freezing.")
        models[agent] = model
        report[agent] = {
            "architecture": architecture,
            "output_dimension": output_dimension,
            "strict_load": True,
            "device": str(device),
            "all_parameters_frozen": True,
        }
    return models, report


class RiskDevDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path = self.frame.iloc[index]["image_path"]
        with Image.open(path) as handle:
            image = handle.convert("RGB")
        return self.transform(image), index


In [ ]:
@dataclasses.dataclass
class MetricResult:
    value: float
    valid: bool
    reason: str | None = None


def metric_ok(value):
    value = float(value)
    if not math.isfinite(value):
        return MetricResult(float("nan"), False, "metric_result_is_not_finite")
    return MetricResult(value, True, None)


def metric_undefined(reason):
    return MetricResult(float("nan"), False, str(reason))


def json_sanitize(value):
    if dataclasses.is_dataclass(value):
        return json_sanitize(dataclasses.asdict(value))
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_sanitize(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_sanitize(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_sanitize(value.tolist())
    if isinstance(value, np.generic):
        return json_sanitize(value.item())
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    return value


def write_json(path, payload):
    sanitized = json_sanitize(payload)
    with Path(path).open("w", encoding="utf-8") as handle:
        json.dump(sanitized, handle, indent=2, sort_keys=True, allow_nan=False)
        handle.write("\n")


def entropy_rows(probabilities):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    terms = np.zeros_like(probabilities)
    positive = probabilities > 0
    terms[positive] = probabilities[positive] * np.log(probabilities[positive])
    return -terms.sum(axis=1)


def safe_macro_auc(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    observed = sorted(map(int, np.unique(labels)))
    missing = sorted(set(range(NUM_CLASSES)) - set(observed))
    if missing:
        return metric_undefined(f"macro_roc_auc_requires_all_classes; missing={missing}")
    try:
        return metric_ok(roc_auc_score(labels, probabilities, labels=list(range(NUM_CLASSES)),
                                       multi_class="ovr", average="macro"))
    except ValueError as exc:
        return metric_undefined(f"macro_roc_auc_undefined: {exc}")


def safe_balanced_accuracy(labels, predictions, context="balanced_accuracy"):
    labels = np.asarray(labels, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    missing = sorted(set(range(NUM_CLASSES)) - set(map(int, np.unique(labels))))
    if missing:
        return metric_undefined(f"{context}_requires_all_classes; missing={missing}")
    recalls = []
    for label in range(NUM_CLASSES):
        mask = labels == label
        recalls.append(float(np.mean(predictions[mask] == label)))
    return metric_ok(np.mean(recalls))


def top_label_adaptive_ece_15(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if len(labels) == 0:
        return metric_undefined("top_label_adaptive_ece_15_requires_nonempty_sample")
    confidence = probabilities.max(axis=1)
    predictions = probabilities.argmax(axis=1)
    correctness = predictions == labels
    order = np.argsort(confidence, kind="mergesort")
    bins = np.array_split(order, min(15, len(order)))
    ece = 0.0
    for indices in bins:
        if len(indices) == 0:
            continue
        gap = abs(float(correctness[indices].mean()) - float(confidence[indices].mean()))
        ece += (len(indices) / len(labels)) * gap
    return metric_ok(ece)


def safe_failure_detection_metric(target, scores, kind):
    target = np.asarray(target, dtype=int)
    observed = sorted(map(int, np.unique(target)))
    if observed != [0, 1]:
        return metric_undefined(f"failure_target_requires_both_0_and_1; observed={observed}")
    try:
        if kind == "auroc":
            return metric_ok(roc_auc_score(target, scores))
        if kind == "auprc":
            return metric_ok(average_precision_score(target, scores))
    except ValueError as exc:
        return metric_undefined(f"failure_detection_{kind}_undefined: {exc}")
    raise ValueError(f"Unknown failure metric: {kind}")


def empirical_aurc(failures, scores, normalized_ids):
    failures = np.asarray(failures, dtype=np.float64)
    scores = np.asarray(scores, dtype=np.float64)
    normalized_ids = np.asarray(normalized_ids, dtype=str)
    if len(failures) == 0:
        return metric_undefined("aurc_requires_nonempty_sample"), np.array([]), np.array([]), np.array([], dtype=int)
    order = np.lexsort((normalized_ids, scores))
    cumulative_errors = np.cumsum(failures[order])
    prefix_sizes = np.arange(1, len(failures) + 1)
    selective_risk = cumulative_errors / prefix_sizes
    coverage = prefix_sizes / len(failures)
    return metric_ok(selective_risk.mean()), coverage, selective_risk, order


def selective_metrics_at_coverage(labels, predictions, order, requested_coverage):
    labels = np.asarray(labels, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    retained_n = max(1, int(math.ceil(float(requested_coverage) * len(labels))))
    retained = np.asarray(order[:retained_n], dtype=int)
    y_true = labels[retained]
    y_pred = predictions[retained]
    per_class = {}
    absent = []
    recalls = []
    for label, name in enumerate(EXPECTED_CLASSES):
        full_count = int(np.sum(labels == label))
        kept_mask = y_true == label
        kept_count = int(kept_mask.sum())
        if kept_count == 0:
            recall = metric_undefined(f"selective_recall_has_no_retained_{name}_samples")
            absent.append(name)
        else:
            recall = metric_ok(np.mean(y_pred[kept_mask] == label))
            recalls.append(recall.value)
        per_class[name] = {
            "retained_count": kept_count,
            "retained_coverage": (kept_count / full_count) if full_count else None,
            "selective_recall": recall,
        }
    if absent:
        balanced = metric_undefined(
            "selective_balanced_accuracy_requires_all_ground_truth_classes; absent=" + repr(absent)
        )
    else:
        balanced = metric_ok(np.mean(recalls))
    return {
        "requested_coverage": float(requested_coverage),
        "actual_coverage": retained_n / len(labels),
        "retained_count": retained_n,
        "ordinary_selective_risk": float(np.mean(y_pred != y_true)),
        "selective_balanced_accuracy": balanced,
        "absent_classes": absent,
        "per_class": per_class,
    }


def classification_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    predictions = probabilities.argmax(axis=1)
    if len(labels) == 0:
        empty = metric_undefined("classification_metrics_require_nonempty_sample")
        return {name: empty for name in ("accuracy", "balanced_accuracy", "macro_roc_auc_ovr",
                                          "macro_f1", "weighted_f1", "negative_log_likelihood",
                                          "multiclass_brier_score", "top_label_adaptive_ece_15")}
    true_probability = np.clip(probabilities[np.arange(len(labels)), labels], 1e-15, 1.0)
    one_hot = np.eye(NUM_CLASSES, dtype=np.float64)[labels]
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predictions, labels=list(range(NUM_CLASSES)), zero_division=0
    )
    return {
        "accuracy": metric_ok(accuracy_score(labels, predictions)),
        "balanced_accuracy": safe_balanced_accuracy(labels, predictions),
        "macro_roc_auc_ovr": safe_macro_auc(labels, probabilities),
        "macro_f1": metric_ok(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)),
                                             average="macro", zero_division=0)),
        "weighted_f1": metric_ok(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)),
                                                average="weighted", zero_division=0)),
        "negative_log_likelihood": metric_ok(-np.log(true_probability).mean()),
        "multiclass_brier_score": metric_ok(np.square(probabilities - one_hot).sum(axis=1).mean()),
        "top_label_adaptive_ece_15": top_label_adaptive_ece_15(labels, probabilities),
        "per_class": {
            EXPECTED_CLASSES[index]: {
                "precision": float(precision[index]), "recall": float(recall[index]),
                "f1": float(f1[index]), "support": int(support[index]),
            }
            for index in range(NUM_CLASSES)
        },
        "confusion_matrix": confusion_matrix(
            labels, predictions, labels=list(range(NUM_CLASSES))
        ).astype(int).tolist(),
    }


In [ ]:
def build_prediction_frame(frame, logits_a, logits_b):
    logits_a = np.asarray(logits_a, dtype=np.float64)
    logits_b = np.asarray(logits_b, dtype=np.float64)
    if logits_a.shape != (len(frame), NUM_CLASSES) or logits_b.shape != (len(frame), NUM_CLASSES):
        raise RuntimeError("Agent logit arrays do not match the risk_dev row contract.")
    probabilities_a = torch.softmax(torch.from_numpy(logits_a), dim=1).numpy()
    probabilities_b = torch.softmax(torch.from_numpy(logits_b), dim=1).numpy()
    probabilities_ensemble = 0.5 * (probabilities_a + probabilities_b)

    for name, probabilities in (("agent_a", probabilities_a), ("agent_b", probabilities_b),
                                ("ensemble", probabilities_ensemble)):
        if not np.isfinite(probabilities).all():
            raise RuntimeError(f"{name} probabilities contain NaN or infinity.")
        if (probabilities < 0).any():
            raise RuntimeError(f"{name} probabilities contain negative values.")
        if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-8, rtol=0.0):
            raise RuntimeError(f"{name} probability vectors do not sum to one.")
    if not np.allclose(probabilities_ensemble, (probabilities_a + probabilities_b) / 2.0,
                       atol=0.0, rtol=0.0):
        raise RuntimeError("Ensemble probabilities are not the exact arithmetic mean.")

    entropy_a = entropy_rows(probabilities_a)
    entropy_b = entropy_rows(probabilities_b)
    entropy_ensemble = entropy_rows(probabilities_ensemble)
    log_mean = np.log(probabilities_ensemble)
    kl_a_terms = np.zeros_like(probabilities_a)
    kl_b_terms = np.zeros_like(probabilities_b)
    positive_a = probabilities_a > 0
    positive_b = probabilities_b > 0
    kl_a_terms[positive_a] = probabilities_a[positive_a] * (
        np.log(probabilities_a[positive_a]) - log_mean[positive_a]
    )
    kl_b_terms[positive_b] = probabilities_b[positive_b] * (
        np.log(probabilities_b[positive_b]) - log_mean[positive_b]
    )
    js = 0.5 * kl_a_terms.sum(axis=1) + 0.5 * kl_b_terms.sum(axis=1)
    identity_rhs = 0.5 * entropy_a + 0.5 * entropy_b + js
    identity_error = np.abs(entropy_ensemble - identity_rhs)
    max_identity_error = float(identity_error.max(initial=0.0))
    if max_identity_error > 1e-6:
        raise RuntimeError(f"Entropy/JS identity failed: maximum error={max_identity_error}")

    labels = frame["label"].to_numpy(dtype=int)
    pred_a = probabilities_a.argmax(axis=1)
    pred_b = probabilities_b.argmax(axis=1)
    pred_ensemble = probabilities_ensemble.argmax(axis=1)
    sorted_probabilities = np.sort(probabilities_ensemble, axis=1)
    top1 = sorted_probabilities[:, -1]
    top2 = sorted_probabilities[:, -2]
    margin = top1 - top2

    result = frame[["image", "image_path", "lesion_id", "lesion_group", "label",
                    "_normalized_image_id"]].copy()
    result["label_name"] = [EXPECTED_CLASSES[value] for value in labels]
    for index, class_name in enumerate(EXPECTED_CLASSES):
        result[f"agent_a_logit_{class_name}"] = logits_a[:, index]
        result[f"agent_a_probability_{class_name}"] = probabilities_a[:, index]
        result[f"agent_b_logit_{class_name}"] = logits_b[:, index]
        result[f"agent_b_probability_{class_name}"] = probabilities_b[:, index]
        result[f"ensemble_probability_{class_name}"] = probabilities_ensemble[:, index]
    for prefix, predictions, probabilities, entropy in (
        ("agent_a", pred_a, probabilities_a, entropy_a),
        ("agent_b", pred_b, probabilities_b, entropy_b),
        ("ensemble", pred_ensemble, probabilities_ensemble, entropy_ensemble),
    ):
        result[f"{prefix}_predicted_class"] = predictions
        result[f"{prefix}_predicted_class_name"] = [EXPECTED_CLASSES[value] for value in predictions]
        result[f"{prefix}_confidence"] = probabilities.max(axis=1)
        result[f"{prefix}_correct"] = predictions == labels
        result[f"{prefix}_entropy"] = entropy
    result["normalized_ensemble_entropy"] = entropy_ensemble / math.log(NUM_CLASSES)
    result["ensemble_top1_probability"] = top1
    result["ensemble_top2_probability"] = top2
    result["ensemble_top1_minus_top2_margin"] = margin
    result["agents_agree"] = pred_a == pred_b
    result["confidence_difference"] = np.abs(probabilities_a.max(axis=1) - probabilities_b.max(axis=1))
    result["maximum_absolute_probability_difference"] = np.max(
        np.abs(probabilities_a - probabilities_b), axis=1
    )
    result["jensen_shannon_divergence"] = js
    result["normalized_js"] = js / math.log(2.0)
    result["ensemble_failure"] = (pred_ensemble != labels).astype(int)
    result["risk_msp"] = 1.0 - top1
    result["risk_normalized_entropy"] = result["normalized_ensemble_entropy"]
    result["risk_margin"] = 1.0 - margin
    result["risk_normalized_js"] = result["normalized_js"]
    result["risk_dual_uncertainty"] = np.maximum(
        result["risk_normalized_entropy"], result["risk_normalized_js"]
    )
    validation = {
        "rows_expected": len(frame), "rows_written": len(result),
        "unique_normalized_images": int(result["_normalized_image_id"].nunique()),
        "arithmetic_probability_mean_exact": True,
        "probability_checks_passed": True,
        "mathematical_identity_maximum_error": max_identity_error,
    }
    if len(result) != len(frame) or result["_normalized_image_id"].duplicated().any():
        raise RuntimeError("Prediction output does not reconstruct risk_dev rows exactly once.")
    return result, validation


def run_frozen_inference(models, frame, device):
    dataset = RiskDevDataset(frame, get_eval_transform())
    loader = DataLoader(dataset, batch_size=(2 if SMOKE_TEST else 32), shuffle=False,
                        num_workers=0, pin_memory=(device.type == "cuda"))
    logits = {"agent_a": [], "agent_b": []}
    expected_index = 0
    with torch.inference_mode():
        for images, indices in loader:
            indices_np = indices.numpy()
            if not np.array_equal(indices_np, np.arange(expected_index, expected_index + len(indices_np))):
                raise RuntimeError("Inference DataLoader changed locked row ordering.")
            expected_index += len(indices_np)
            images = images.to(device, non_blocking=(device.type == "cuda"))
            for agent, model in models.items():
                logits[agent].append(model(images).detach().cpu().numpy())
    return np.concatenate(logits["agent_a"]), np.concatenate(logits["agent_b"])


def risk_score_analysis(predictions):
    labels = predictions["label"].to_numpy(dtype=int)
    ensemble_predictions = predictions["ensemble_predicted_class"].to_numpy(dtype=int)
    failures = predictions["ensemble_failure"].to_numpy(dtype=int)
    normalized_ids = predictions["_normalized_image_id"].astype(str).to_numpy()
    records = []
    structured = {}
    for score_name in RISK_SCORE_COLUMNS:
        scores = predictions[score_name].to_numpy(dtype=float)
        aurc, coverage_curve, risk_curve, order = empirical_aurc(failures, scores, normalized_ids)
        auroc = safe_failure_detection_metric(failures, scores, "auroc")
        auprc = safe_failure_detection_metric(failures, scores, "auprc")
        coverage_results = []
        for target in COVERAGE_TARGETS:
            item = selective_metrics_at_coverage(labels, ensemble_predictions, order, target)
            coverage_results.append(item)
            flat = {
                "risk_score": score_name,
                "failure_detection_auroc": auroc.value,
                "failure_detection_auroc_valid": auroc.valid,
                "failure_detection_auroc_reason": auroc.reason,
                "failure_detection_auprc": auprc.value,
                "failure_detection_auprc_valid": auprc.valid,
                "failure_detection_auprc_reason": auprc.reason,
                "empirical_aurc": aurc.value,
                "coverage_requested": item["requested_coverage"],
                "coverage_actual": item["actual_coverage"],
                "retained_count": item["retained_count"],
                "ordinary_selective_risk": item["ordinary_selective_risk"],
                "selective_balanced_accuracy": item["selective_balanced_accuracy"].value,
                "selective_balanced_accuracy_valid": item["selective_balanced_accuracy"].valid,
                "selective_balanced_accuracy_reason": item["selective_balanced_accuracy"].reason,
                "absent_classes": ";".join(item["absent_classes"]),
            }
            for class_name, values in item["per_class"].items():
                flat[f"retained_count_{class_name}"] = values["retained_count"]
                flat[f"retained_coverage_{class_name}"] = values["retained_coverage"]
                flat[f"selective_recall_{class_name}"] = values["selective_recall"].value
            records.append(flat)
        structured[score_name] = {
            "failure_detection_auroc": auroc,
            "failure_detection_auprc": auprc,
            "empirical_aurc": aurc,
            "coverages": coverage_results,
            "curve": {"coverage": coverage_curve, "ordinary_selective_risk": risk_curve},
        }
    return pd.DataFrame(records), structured


def _indices_for_sampled_groups(group_to_indices, sampled_groups):
    blocks = [np.asarray(group_to_indices[group], dtype=int) for group in sampled_groups]
    return np.concatenate(blocks) if blocks else np.array([], dtype=int)


def paired_cluster_bootstrap(predictions, replicates=BOOTSTRAP_REPLICATES, seed=BOOTSTRAP_SEED):
    groups = predictions["lesion_group"].astype(str).to_numpy()
    unique_groups = np.array(sorted(set(groups)), dtype=object)
    group_to_indices = {group: np.flatnonzero(groups == group) for group in unique_groups}
    rng = np.random.default_rng(seed)
    probabilities = {
        model: predictions[[f"{model}_probability_{name}" for name in EXPECTED_CLASSES]].to_numpy(float)
        for model in ("agent_a", "agent_b", "ensemble")
    }
    labels = predictions["label"].to_numpy(int)
    normalized_ids = predictions["_normalized_image_id"].astype(str).to_numpy()
    values = defaultdict(list)
    invalid = defaultdict(Counter)

    def collect(key, result):
        if result.valid:
            values[key].append(float(result.value))
        else:
            invalid[key][result.reason or "undefined_without_reason"] += 1

    for _ in range(int(replicates)):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = _indices_for_sampled_groups(group_to_indices, sampled_groups)
        replicate_metrics = {}
        for model in ("agent_a", "agent_b", "ensemble"):
            model_metrics = classification_metrics(labels[indices], probabilities[model][indices])
            replicate_metrics[model] = model_metrics
            for metric_name in ("accuracy", "balanced_accuracy", "macro_roc_auc_ovr", "macro_f1",
                                "weighted_f1", "negative_log_likelihood", "multiclass_brier_score",
                                "top_label_adaptive_ece_15"):
                collect(f"classification.{model}.{metric_name}", model_metrics[metric_name])
        for baseline in ("agent_a", "agent_b"):
            for metric_name in ("accuracy", "balanced_accuracy", "macro_roc_auc_ovr", "macro_f1",
                                "weighted_f1", "negative_log_likelihood", "multiclass_brier_score",
                                "top_label_adaptive_ece_15"):
                ensemble_value = replicate_metrics["ensemble"][metric_name]
                baseline_value = replicate_metrics[baseline][metric_name]
                key = f"paired.ensemble_minus_{baseline}.{metric_name}"
                if ensemble_value.valid and baseline_value.valid:
                    collect(key, metric_ok(ensemble_value.value - baseline_value.value))
                else:
                    reasons = [result.reason for result in (ensemble_value, baseline_value) if not result.valid]
                    collect(key, metric_undefined("paired_metric_invalid: " + " | ".join(reasons)))

        subset = predictions.iloc[indices]
        failures = subset["ensemble_failure"].to_numpy(int)
        for score_name in RISK_SCORE_COLUMNS:
            scores = subset[score_name].to_numpy(float)
            aurc, _coverage, _risk, _order = empirical_aurc(
                failures, scores, normalized_ids[indices]
            )
            collect(f"risk.{score_name}.empirical_aurc", aurc)
            collect(f"risk.{score_name}.failure_detection_auroc",
                    safe_failure_detection_metric(failures, scores, "auroc"))
            collect(f"risk.{score_name}.failure_detection_auprc",
                    safe_failure_detection_metric(failures, scores, "auprc"))

    summary = {}
    all_keys = sorted(set(values) | set(invalid))
    for key in all_keys:
        finite_values = np.asarray(values[key], dtype=float)
        summary[key] = {
            "valid_replicates": int(len(finite_values)),
            "invalid_replicates": int(sum(invalid[key].values())),
            "invalid_reason_counts": dict(invalid[key]),
            "percentile_95_ci": (
                [float(np.percentile(finite_values, 2.5)), float(np.percentile(finite_values, 97.5))]
                if len(finite_values) else None
            ),
            "bootstrap_mean": float(finite_values.mean()) if len(finite_values) else None,
        }
    return {
        "sampling_unit": "lesion_group",
        "seed": int(seed),
        "requested_replicates": int(replicates),
        "unique_lesion_groups": int(len(unique_groups)),
        "paired_resamples": True,
        "repeated_sampled_groups_preserved_as_repeated_clusters": True,
        "metrics": summary,
    }


In [ ]:
def _plot_confusion(matrix, title, path):
    matrix = np.asarray(matrix, dtype=int)
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
           xticklabels=EXPECTED_CLASSES, yticklabels=EXPECTED_CLASSES,
           xlabel="Predicted", ylabel="True", title=title)
    for row in range(NUM_CLASSES):
        for column in range(NUM_CLASSES):
            ax.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def create_figures(predictions, classification, risk_structured, figure_dir):
    figure_dir.mkdir(parents=True, exist_ok=True)
    figure_paths = {}
    for model, filename in (("agent_a", "confusion_agent_a.png"),
                            ("agent_b", "confusion_agent_b.png"),
                            ("ensemble", "confusion_ensemble.png")):
        path = figure_dir / filename
        _plot_confusion(classification[model]["confusion_matrix"], f"{model} confusion matrix", path)
        figure_paths[filename] = path

    fig, ax = plt.subplots(figsize=(8, 6))
    for score_name, values in risk_structured.items():
        ax.plot(values["curve"]["coverage"], values["curve"]["ordinary_selective_risk"], label=score_name)
    ax.set(xlabel="Coverage", ylabel="Ordinary selective 0-1 risk",
           title="Risk-coverage candidates")
    ax.legend(fontsize=7)
    fig.tight_layout()
    path = figure_dir / "risk_coverage_candidates.png"
    fig.savefig(path, dpi=150); plt.close(fig); figure_paths[path.name] = path

    failures = predictions["ensemble_failure"].to_numpy(int)
    for plot_kind, filename in (("pr", "failure_detection_pr.png"), ("roc", "failure_detection_roc.png")):
        fig, ax = plt.subplots(figsize=(8, 6))
        plotted = False
        if sorted(map(int, np.unique(failures))) == [0, 1]:
            for score_name in RISK_SCORE_COLUMNS:
                scores = predictions[score_name].to_numpy(float)
                if plot_kind == "pr":
                    y, x, _ = precision_recall_curve(failures, scores)
                else:
                    x, y, _ = roc_curve(failures, scores)
                ax.plot(x, y, label=score_name); plotted = True
        if not plotted:
            ax.text(0.5, 0.5, "Undefined: failure target has one class", ha="center", va="center")
        ax.set(xlabel=("Recall" if plot_kind == "pr" else "False-positive rate"),
               ylabel=("Precision" if plot_kind == "pr" else "True-positive rate"),
               title=("Failure-detection precision-recall" if plot_kind == "pr" else "Failure-detection ROC"))
        if plotted: ax.legend(fontsize=7)
        fig.tight_layout(); path = figure_dir / filename
        fig.savefig(path, dpi=150); plt.close(fig); figure_paths[path.name] = path

    probabilities = predictions[[f"ensemble_probability_{name}" for name in EXPECTED_CLASSES]].to_numpy(float)
    labels = predictions["label"].to_numpy(int)
    confidence = probabilities.max(axis=1)
    correct = probabilities.argmax(axis=1) == labels
    order = np.argsort(confidence, kind="mergesort")
    bins = np.array_split(order, min(15, len(order)))
    bin_conf = [float(confidence[index].mean()) for index in bins if len(index)]
    bin_acc = [float(correct[index].mean()) for index in bins if len(index)]
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot([0, 1], [0, 1], "--", color="black", linewidth=1)
    ax.plot(bin_conf, bin_acc, "o-")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Top-label confidence", ylabel="Accuracy",
           title="Raw top-label adaptive reliability")
    fig.tight_layout(); path = figure_dir / "raw_reliability_diagram.png"
    fig.savefig(path, dpi=150); plt.close(fig); figure_paths[path.name] = path
    return figure_paths


def _safe_metadata_for_manifest(metadata):
    def clean(value):
        if isinstance(value, dict):
            return {str(key): clean(item) for key, item in value.items()
                    if FORBIDDEN_SPLIT_TOKEN not in str(key).lower()}
        if isinstance(value, list):
            return [clean(item) for item in value]
        if isinstance(value, str) and FORBIDDEN_SPLIT_TOKEN in value.lower():
            return "[forbidden locked-test reference omitted]"
        return value
    return clean(metadata)


def write_outputs(predictions, classification, risk_table, risk_structured, bootstrap,
                  validation_context, device):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    figure_dir = OUTPUT_DIR / "figures"
    paths = {
        "predictions_csv": OUTPUT_DIR / "risk_dev_predictions.csv",
        "predictions_parquet": OUTPUT_DIR / "risk_dev_predictions.parquet",
        "classification_metrics": OUTPUT_DIR / "risk_dev_classification_metrics.json",
        "risk_score_summary": OUTPUT_DIR / "risk_dev_risk_score_summary.csv",
        "bootstrap_results": OUTPUT_DIR / "risk_dev_bootstrap_results.json",
        "manifest": OUTPUT_DIR / "risk_dev_manifest.json",
    }
    for path in paths.values():
        _assert_not_forbidden_path(path, "Stage 5 output")
    predictions.to_csv(paths["predictions_csv"], index=False)
    predictions.to_parquet(paths["predictions_parquet"], index=False)
    write_json(paths["classification_metrics"], classification)
    risk_table.to_csv(paths["risk_score_summary"], index=False)
    write_json(paths["bootstrap_results"], bootstrap)
    figures = create_figures(predictions, classification, risk_structured, figure_dir)

    output_hashes = {}
    for path in [*paths.values(), *figures.values()]:
        if path == paths["manifest"]:
            continue
        output_hashes[str(path.relative_to(OUTPUT_DIR))] = _sha256_file(path)
    manifest = {
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "git_commit": validation_context["git_commit"],
        "smoke_test": SMOKE_TEST,
        "smoke_metrics_are_scientific_results": False if SMOKE_TEST else None,
        "smoke_warning": (
            "Diagnostic subset and 50 bootstrap replicates must not be interpreted or reported as scientific results."
            if SMOKE_TEST else None
        ),
        "package_compatibility": PACKAGE_COMPATIBILITY,
        "recorded_training_package_versions": RECORDED_PACKAGE_VERSIONS,
        "device": {"selected": str(device), "cuda_available": torch.cuda.is_available(),
                   "cuda_version": torch.version.cuda},
        "random_seed": BOOTSTRAP_SEED,
        "paths": validation_context["paths"],
        "input_hashes": validation_context["input_hashes"],
        "output_hashes_excluding_manifest": output_hashes,
        "class_order": EXPECTED_CLASSES,
        "class_order_provenance": validation_context["class_order_provenance"],
        "preprocessing": {
            "image_size": IMAGE_SIZE, "resize_short_side": 256,
            "mean": IMAGENET_MEAN, "std": IMAGENET_STD,
            "transform": repr(get_eval_transform()),
        },
        "architectures": validation_context["model_validation"],
        "output_schema": {"columns": list(predictions.columns),
                          "dtypes": {name: str(dtype) for name, dtype in predictions.dtypes.items()}},
        "number_of_samples": len(predictions),
        "lesion_group_count": int(predictions["lesion_group"].nunique()),
        "class_counts": {EXPECTED_CLASSES[i]: int(np.sum(predictions["label"] == i))
                         for i in range(NUM_CLASSES)},
        "checkpoint_metadata": {agent: _safe_metadata_for_manifest(metadata)
                                for agent, metadata in CHECKPOINT_METADATA.items()},
        "validation_results": validation_context["validation_results"],
        "mathematical_identity_maximum_error": validation_context["prediction_validation"]["mathematical_identity_maximum_error"],
        "metric_definitions": {
            "top_label_adaptive_ece_15": "Top-class confidence/correctness, stable confidence sort, 15 approximately equal-mass bins, sample-weighted absolute gap.",
            "empirical_aurc": "Sort lowest-to-highest risk with normalized image ID tie-break; for k=1..N use prefix coverage k/N and ordinary retained 0-1 error; AURC is the mean of all N prefix risks.",
            "cluster_bootstrap": "Sample lesion_group with replacement, include every image row, preserve repeated groups, and use identical resamples for both agents and ensemble.",
        },
        "locked_test_accessed": False,
        "learned_consensus_model_used": False,
        "threshold_selected": False,
        "calibrator_fitted": False,
        "explainability_executed": False,
        "manifest_self_hash_omitted_to_avoid_recursive_hashing": True,
    }
    write_json(paths["manifest"], manifest)
    return paths, figures, _sha256_file(paths["manifest"])


In [ ]:
def run_synthetic_helper_tests():
    # Undefined binary metrics remain invalid and serialize to strict JSON null.
    undefined = safe_failure_detection_metric(np.zeros(4, dtype=int), np.linspace(0, 1, 4), "auroc")
    assert not undefined.valid and math.isnan(undefined.value) and undefined.reason
    payload = json_sanitize({"metric": undefined, "nan": float("nan"), "infinity": float("inf")})
    encoded = json.dumps(payload, allow_nan=False)
    assert '"value": null' in encoded and '"nan": null' in encoded and '"infinity": null' in encoded

    # Exact empirical AURC: sorted failures [0,1,1] -> prefix risks [0, 1/2, 2/3].
    aurc, coverage, selective_risk, order = empirical_aurc(
        np.array([1, 0, 1]), np.array([0.2, 0.1, 0.3]), np.array(["b", "a", "c"])
    )
    assert aurc.valid and np.isclose(aurc.value, np.mean([0.0, 0.5, 2.0 / 3.0]))
    assert np.array_equal(order, np.array([1, 0, 2])) and np.isclose(coverage[-1], 1.0)

    # Adaptive ECE and missing-class selective balanced accuracy behavior.
    labels = np.array([0, 1, 0, 1])
    probs = np.array([[0.8, 0.2, 0, 0, 0, 0, 0, 0], [0.3, 0.7, 0, 0, 0, 0, 0, 0],
                      [0.6, 0.4, 0, 0, 0, 0, 0, 0], [0.9, 0.1, 0, 0, 0, 0, 0, 0]], dtype=float)
    assert top_label_adaptive_ece_15(labels, probs).valid
    selective = selective_metrics_at_coverage(labels, probs.argmax(axis=1), np.arange(4), 0.5)
    assert not selective["selective_balanced_accuracy"].valid and selective["absent_classes"]
    assert not safe_macro_auc(labels, probs).valid

    # Repeated cluster draws produce repeated row blocks, never image resampling.
    mapping = {"L:a": np.array([0, 1]), "L:b": np.array([2])}
    repeated = _indices_for_sampled_groups(mapping, ["L:a", "L:a", "L:b"])
    assert np.array_equal(repeated, np.array([0, 1, 0, 1, 2]))

    # Mixed direct/nested image resolution and duplicate detection.
    with tempfile.TemporaryDirectory(prefix="argus_stage5_image_test_") as tmp:
        root = Path(tmp); nested = root / "nested"; nested.mkdir()
        (root / "ISIC_A.JPG").touch(); (nested / "ISIC_B.png").touch()
        frame = pd.DataFrame({"_normalized_image_id": ["isic_a", "isic_b"]})
        resolved = resolve_image_paths(frame, root)
        assert resolved["image_path"].notna().all()
        (nested / "ISIC_B.jpeg").touch()
        try:
            resolve_image_paths(frame, root)
        except RuntimeError as exc:
            assert "multiple files" in str(exc)
        else:
            raise AssertionError("Duplicate normalized image stem did not fail.")

    # Prediction schema and mathematical checks use logits only, never images.
    synthetic_frame = pd.DataFrame({
        "image": [f"ISIC_{index}" for index in range(8)],
        "image_path": [f"unused_{index}.jpg" for index in range(8)],
        "lesion_id": [f"L{index}" for index in range(8)],
        "lesion_group": [f"L:L{index}" for index in range(8)],
        "label": list(range(8)),
        "_normalized_image_id": [f"isic_{index}" for index in range(8)],
    })
    logits_a = np.eye(8) * 2.0
    logits_b = np.eye(8) * 1.5
    prediction_frame, checks = build_prediction_frame(synthetic_frame, logits_a, logits_b)
    assert len(prediction_frame) == 8 and checks["arithmetic_probability_mean_exact"]
    required = {"agent_a_logit_MEL", "agent_b_probability_SCC", "ensemble_probability_NV",
                "ensemble_failure", *RISK_SCORE_COLUMNS}
    assert required.issubset(prediction_frame.columns)
    bootstrap = paired_cluster_bootstrap(prediction_frame, replicates=3, seed=BOOTSTRAP_SEED)
    failure_key = "risk.risk_msp.failure_detection_auroc"
    assert bootstrap["metrics"][failure_key]["valid_replicates"] == 0
    assert bootstrap["metrics"][failure_key]["invalid_replicates"] == 3
    assert bootstrap["metrics"][failure_key]["invalid_reason_counts"]
    return {
        "undefined_metric_serialization": True,
        "empirical_aurc_definition": True,
        "top_label_adaptive_ece_15": True,
        "selective_missing_class_handling": True,
        "paired_repeated_cluster_indices": True,
        "mixed_direct_nested_image_resolution": True,
        "duplicate_image_detection": True,
        "prediction_schema_and_identity": True,
        "paired_bootstrap_undefined_metric_accounting": True,
    }


def run_static_notebook_checks(notebook_path):
    notebook = json.loads(Path(notebook_path).read_text(encoding="utf-8"))
    sources = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") == "code":
            source = cell.get("source", "")
            sources.append("".join(source) if isinstance(source, list) else source)
    source = "\n".join(sources)
    tree = ast.parse(source)
    forbidden_imports = {"light" + "gbm", "xg" + "boost"}
    imported = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            imported.update(alias.name.split(".")[0] for alias in node.names)
        elif isinstance(node, ast.ImportFrom) and node.module:
            imported.add(node.module.split(".")[0])
    assert not (imported & forbidden_imports), f"Forbidden imports: {imported & forbidden_imports}"
    assert ("split_" + "summary.json") not in source
    assert ("strict" + "=False") not in source
    assert ("pretrained" + "=True") not in source
    assert "probabilities_ensemble = 0.5 * (probabilities_a + probabilities_b)" in source
    assert ("Grad" + "CAM") not in source and ("attention_" + "rollout") not in source
    forbidden_symbols = {"CalibratedClassifierCV", "TemperatureScaler", "optimize_threshold",
                         "fit_calibrator", "train_consensus", "ConsensusMLP"}
    referenced_names = {node.id for node in ast.walk(tree) if isinstance(node, ast.Name)}
    assert not (referenced_names & forbidden_symbols)
    assert "allow_nan=False" in source
    return {
        "forbidden_imports_absent": True,
        "split_summary_not_read": True,
        "strict_checkpoint_loading_required": True,
        "no_pretrained_fallback": True,
        "probability_averaging_contract_present": True,
        "explainability_absent": True,
        "calibration_threshold_and_learned_consensus_absent": True,
        "strict_json_serialization_present": True,
    }


def _git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT,
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


def run_pre_inference_validation(local_cpu=False):
    notebook_path = REPO_ROOT / "ml_training" / "03_export_risk_dev_predictions.ipynb"
    synthetic = run_synthetic_helper_tests()
    static = run_static_notebook_checks(notebook_path)
    with materialize_split_source(SPLIT_SOURCE, SPLIT_SOURCE_KIND) as split_paths:
        risk, split_frames, fingerprint, fingerprint_hash, split_hashes, overlap_report = (
            _load_and_validate_splits(split_paths, CHECKPOINT_METADATA)
        )
    class_provenance = verify_class_order_provenance(CHECKPOINT_METADATA, risk["label"].to_numpy())
    extraction_root = None
    with extracted_checkpoint_inputs(CHECKPOINT_PATHS) as (extracted, member_hashes, temp_root):
        extraction_root = Path(temp_root)
        device = torch.device("cpu") if local_cpu else torch.device("cuda" if torch.cuda.is_available() else "cpu")
        models, model_validation = reconstruct_and_load_models(extracted, device=device)
    if extraction_root is None or extraction_root.exists():
        raise RuntimeError("Temporary checkpoint extraction directory was not cleaned.")
    context = {
        "risk_dev": risk,
        "split_frames": split_frames,
        "models": models,
        "device": device,
        "git_commit": _git_commit(),
        "class_order_provenance": class_provenance,
        "model_validation": model_validation,
        "paths": {
            "repository_root": str(REPO_ROOT), "checkpoint_directory": str(CHECKPOINT_DIR),
            "split_source": str(SPLIT_SOURCE), "artifact_root": str(ARTIFACT_ROOT),
            "image_directory": IMAGE_DIR_ENV or None, "output_directory": str(OUTPUT_DIR),
        },
        "input_hashes": {
            "checkpoint_archives": ARCHIVE_HASHES,
            "checkpoint_members": member_hashes,
            "split_csvs": split_hashes,
            "dataset_fingerprint_json": fingerprint_hash,
        },
        "validation_results": {
            "synthetic_helper_tests": synthetic,
            "static_notebook_checks": static,
            "split_overlap_counts": overlap_report,
            "verified_lesion_group_contract": True,
            "checkpoint_temporary_cleanup": True,
            "metadata_fingerprint_compatibility": True,
            "strict_checkpoint_loading": True,
        },
    }
    return context


def run_stage5():
    context = run_pre_inference_validation(local_cpu=False)
    if not IMAGE_DIR_ENV:
        raise RuntimeError("ARGUS_IMAGE_DIR is required before inference; no image path will be guessed.")
    frame = select_smoke_subset(context["risk_dev"]) if SMOKE_TEST else context["risk_dev"].copy()
    frame = resolve_image_paths(frame, IMAGE_DIR_ENV)
    logits_a, logits_b = run_frozen_inference(context["models"], frame, context["device"])
    predictions, prediction_validation = build_prediction_frame(frame, logits_a, logits_b)
    context["prediction_validation"] = prediction_validation
    probability_columns = {
        model: [f"{model}_probability_{name}" for name in EXPECTED_CLASSES]
        for model in ("agent_a", "agent_b", "ensemble")
    }
    classification = {
        model: classification_metrics(predictions["label"].to_numpy(int),
                                      predictions[columns].to_numpy(float))
        for model, columns in probability_columns.items()
    }
    risk_table, risk_structured = risk_score_analysis(predictions)
    bootstrap = paired_cluster_bootstrap(predictions)
    paths, figures, manifest_hash = write_outputs(
        predictions, classification, risk_table, risk_structured, bootstrap, context, context["device"]
    )
    print("Stage 5 completed:", len(predictions), "rows")
    print("Outputs:", OUTPUT_DIR)
    print("Manifest SHA-256:", manifest_hash)
    return {"paths": paths, "figures": figures, "manifest_sha256": manifest_hash}


In [ ]:
# Execute the complete measurement only when this notebook is run normally.
# Local repository validation invokes run_pre_inference_validation(local_cpu=True)
# without executing this cell, so no local image inference can occur.
STAGE5_RESULT = run_stage5()
